## torch.autograd.grad() 

In [150]:
import sys
import torch
import torch.nn as nn 
import torch.optim as optim

In [144]:
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 =nn.Linear(2,2)
        self.fc2 =nn.Linear(2,1)

    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x 

In [145]:
data = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
target = torch.tensor([[0.0], [1.0], [1.0], [0.0]])


In [155]:
model = SimpleNN()
criterion = nn.MSELoss()
# optimizer = optim.SGD(model.parameters(), lr = 0.1)


for epoch in range(1000):
    for i in range(len(data)):
        x = data[i]
        y = target[i]
        # optimizer.zero_grad()

        prediction =  model(x)
        loss = criterion(prediction, y)
        # loss.backward()
        grads = torch.autograd.grad(loss, model.parameters(), retain_graph=True)

        with torch.no_grad():
            for param, grad in zip(model.parameters(), grads):
                print(param)
                print("----")
                print("grad is: ")
                print(grad)
                param -= grad * 0.1
                print("-----------------------------------------------")
        # optimizer.step()
        if epoch % 100 == 0:
          print(f'Epoch [{epoch+1}/1000], Loss: {loss.item():.4f}')




Parameter containing:
tensor([[ 0.6475, -0.3976],
        [-0.2076,  0.1435]], requires_grad=True)
----
grad is: 
tensor([[0., 0.],
        [0., 0.]])
-----------------------------------------------
Parameter containing:
tensor([-0.5813, -0.5528], requires_grad=True)
----
grad is: 
tensor([0., 0.])
-----------------------------------------------
Parameter containing:
tensor([[0.3424, 0.6429]], requires_grad=True)
----
grad is: 
tensor([[0., 0.]])
-----------------------------------------------
Parameter containing:
tensor([-0.6440], requires_grad=True)
----
grad is: 
tensor([-1.2881])
-----------------------------------------------
Epoch [1/1000], Loss: 0.4148
Parameter containing:
tensor([[ 0.6475, -0.3976],
        [-0.2076,  0.1435]], requires_grad=True)
----
grad is: 
tensor([[0., 0.],
        [0., 0.]])
-----------------------------------------------
Parameter containing:
tensor([-0.5813, -0.5528], requires_grad=True)
----
grad is: 
tensor([0., 0.])
-------------------------------

In [ ]:
test_input = torch.tensor([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
predictions = model(test_input)
print("Predictions after training:")
print(predictions)

## AR: GD w/o loss.backward

In [105]:
# create you data
import torch
import torch.nn as nn
import torch.optim as optim

In [106]:
#optimizer.step(closure)
#XOR
data = torch.tensor([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
labels = torch.tensor([[0.0], [1.0], [1.0], [0.0]])


In [112]:
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(2,2)
        self.fc2 = nn.Linear(2,1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [113]:
model = SimpleNN()

In [114]:
criterion = nn.MSELoss()

In [131]:
for epoch in range(100):
    for i in range(len(data)):
        x = data[i]
        y = labels[i]

        prediction  = model(x)
        # print(prediction)
        loss = criterion(y, prediction)
        grad  = torch.autograd.grad(loss, model.parameters())
        for param in model.parameters():
            with torch.no_grad():
                print(grad[0])
                print(grad[1])
                print(grad[2])
                print(grad[3])
                print(len(grad))
                print("-----------")
                print(param[0])
                print(param[1])
                print(len(param))
                param -= grad * 0.1

        grad.zero



tensor([[0., 0.],
        [-0., -0.]])
tensor([ 0.0000, -0.2393])
tensor([[ 0.0000, -0.3083]])
tensor([-0.4812])
4
-----------
tensor([0.3483, 0.0352], requires_grad=True)
tensor([0.5711, 0.4931], requires_grad=True)
2


TypeError: can't multiply sequence by non-int of type 'float'

In [72]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.registered_fc = nn.Linear(2, 1)  # This is registered
        self.unregistered_fc = nn.Linear(2, 1)  # Not registered

    def forward(self, x):
        return self.unregistered_fc(self.registered_fc(x))


In [79]:
class MyModelSuper(nn.Module):
    def __init__(self):
        super(MyModelSuper, self).__init__()
        self.registered_fc = nn.Linear(2, 1)  # This is registered
        self.unregistered_fc = nn.Linear(2, 1)  # Not registered

    def forward(self, x):
        return self.unregistered_fc(self.registered_fc(x))


### checking registered parameters

In [80]:
model = MyModel()
print(dict(model.named_parameters()).keys())

dict_keys(['registered_fc.weight', 'registered_fc.bias', 'unregistered_fc.weight', 'unregistered_fc.bias'])


In [82]:
modelSuper = MyModelSuper()
print(dict(modelSuper.named_parameters()).keys())

dict_keys(['registered_fc.weight', 'registered_fc.bias', 'unregistered_fc.weight', 'unregistered_fc.bias'])


In [90]:
modelSuper.to('cuda')

MyModelSuper(
  (registered_fc): Linear(in_features=2, out_features=1, bias=True)
  (unregistered_fc): Linear(in_features=2, out_features=1, bias=True)
)

In [91]:
model.to('cuda')

MyModel(
  (registered_fc): Linear(in_features=2, out_features=1, bias=True)
  (unregistered_fc): Linear(in_features=2, out_features=1, bias=True)
)